# Training RFM Model

## Imports

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import joblib

## Data Directories

In [2]:
DATA_DIR= '..data/raw'
PROCESSED_DATA_DIR= '../data/processed'
MODELS_DIR= '../models'

## Data Loading and Feature Engineering

In [4]:
df= pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'churn_training_data.csv'))

In [5]:
df.head()

,customer_unique_id,last_purchase_date,total_orders,total_spent,avg_review_score
0,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-20 19:14:26,15,879.27,5.0000
1,3e43e6105506432c953e165fb2acf44c,2018-02-27 18:36:39,9,1172.67,2.6429
2,ca77025e7201e3b30c44b472ff346268,2018-06-01 11:38:29,7,1122.72,5.0000
3,1b6c7548a2a1f9037c1fd3ddfed95f33,2018-02-14 13:22:12,7,959.01,5.0000
4,6469f99c1f9dfae7733b25662e7f1782,2018-06-28 00:43:34,7,758.83,5.0000


In [6]:
df.columns

Index(['customer_unique_id', 'last_purchase_date', 'total_orders',
       'total_spent', 'avg_review_score'],
      dtype='object')

In [7]:
df.shape

(93358, 5)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93358 entries, 0 to 93357
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_unique_id  93358 non-null  object 
 1   last_purchase_date  93358 non-null  object 
 2   total_orders        93358 non-null  int64  
 3   total_spent         93358 non-null  float64
 4   avg_review_score    92755 non-null  float64
dtypes: float64(2), int64(1), object(2)
memory usage: 3.6+ MB


In [9]:
df.describe()

,total_orders,total_spent,avg_review_score
count,93358.000000,93358.000000,92755.000000
mean,1.033420,165.916853,4.153360
std,0.209097,227.787005,1.280555
min,1.000000,9.590000,1.000000
25%,1.000000,63.100000,4.000000
50%,1.000000,107.890000,5.000000
75%,1.000000,183.120000,5.000000
max,15.000000,13664.080000,5.000000


In [10]:
# Converting TimeStamp to DateTime:
df['last_purchase_date'] = pd.to_datetime(df['last_purchase_date'])

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93358 entries, 0 to 93357
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_unique_id  93358 non-null  object        
 1   last_purchase_date  93358 non-null  datetime64[ns]
 2   total_orders        93358 non-null  int64         
 3   total_spent         93358 non-null  float64       
 4   avg_review_score    92755 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 3.6+ MB


In [14]:
# Calculating Recency Days:
# We will use Max of last_purchase_date to act as 'today'

today= df['last_purchase_date'].max()
df['recency_days']= (today - df['last_purchase_date']).dt.days

In [17]:
df.head()

,customer_unique_id,last_purchase_date,total_orders,total_spent,avg_review_score,recency_days
0,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-20 19:14:26,15,879.27,5.0000,8
1,3e43e6105506432c953e165fb2acf44c,2018-02-27 18:36:39,9,1172.67,2.6429,182
2,ca77025e7201e3b30c44b472ff346268,2018-06-01 11:38:29,7,1122.72,5.0000,89
3,1b6c7548a2a1f9037c1fd3ddfed95f33,2018-02-14 13:22:12,7,959.01,5.0000,196
4,6469f99c1f9dfae7733b25662e7f1782,2018-06-28 00:43:34,7,758.83,5.0000,62


In [18]:
# Handling Missing Values in avg_review_score:
df['avg_review_score'] = df['avg_review_score'].fillna(df['avg_review_score'].median())

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93358 entries, 0 to 93357
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_unique_id  93358 non-null  object        
 1   last_purchase_date  93358 non-null  datetime64[ns]
 2   total_orders        93358 non-null  int64         
 3   total_spent         93358 non-null  float64       
 4   avg_review_score    93358 non-null  float64       
 5   recency_days        93358 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int64(2), object(1)
memory usage: 4.3+ MB


In [21]:
df.head(10)

,customer_unique_id,last_purchase_date,total_orders,total_spent,avg_review_score,recency_days
0,8d50f5eadf50201ccdcedfb9e2ac8455,2018-08-20 19:14:26,15,879.27,5.0000,8
1,3e43e6105506432c953e165fb2acf44c,2018-02-27 18:36:39,9,1172.67,2.6429,182
2,ca77025e7201e3b30c44b472ff346268,2018-06-01 11:38:29,7,1122.72,5.0000,89
3,1b6c7548a2a1f9037c1fd3ddfed95f33,2018-02-14 13:22:12,7,959.01,5.0000,196
4,6469f99c1f9dfae7733b25662e7f1782,2018-06-28 00:43:34,7,758.83,5.0000,62
5,63cfc61cee11cbe306bff5857d00bfe4,2018-05-28 17:20:02,6,826.32,4.3636,92
6,dc813062e0fc23409cd255f7f53c7074,2018-08-23 00:07:26,6,1033.62,4.3636,6
7,f0e310a6839dce9de1638e0fe5ab282a,2018-04-05 09:04:45,6,540.69,4.5000,146
8,47c1a3033b8b77b3ab6e109eb4d5fdf3,2018-01-24 15:15:26,6,997.32,4.8750,216
9,12f5d6e1cbf93dafd9dcc19095df0b3d,2017-01-05 15:25:10,6,110.72,5.0000,600
